In [1]:
import pandas as pd
import numpy as np
import nltk
import gensim
import re
import string
import tensorflow as tf
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [2]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Richa\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Richa\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Richa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
df = pd.read_json("D:/Richa's files/archive/News_Category_Dataset_v3.json", lines=True)

In [5]:
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [6]:
data = df[['headline', 'category']].copy()

In [7]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [8]:
def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize
    tokens = nltk.word_tokenize(text)
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

# Apply preprocessing
data['tokens'] = data['headline'].apply(preprocess_text)

In [9]:
w2v_model = Word2Vec(sentences=data['tokens'], vector_size=100, window=5, min_count=1, workers=4)

# Function to vectorize a sentence by averaging its word embeddings
def vectorize_text(tokens):
    vector = np.zeros(100)
    count = 0
    for word in tokens:
        if word in w2v_model.wv:
            vector += w2v_model.wv[word]
            count += 1
    if count != 0:
        vector = vector / count
    return vector

data['vector'] = data['tokens'].apply(vectorize_text)

In [10]:
X = np.vstack(data['vector'].values)
y = data['category']

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [11]:
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(100,)))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(len(np.unique(y_encoded)), activation='softmax'))

model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1)

Epoch 1/10
2358/2358 [==============================] - 14s 5ms/step - loss: 2.4492 - accuracy: 0.3754 - val_loss: 2.1549 - val_accuracy: 0.4326
Epoch 2/10
2358/2358 [==============================] - 10s 4ms/step - loss: 2.2634 - accuracy: 0.4134 - val_loss: 2.1045 - val_accuracy: 0.4404
Epoch 3/10
2358/2358 [==============================] - 12s 5ms/step - loss: 2.2167 - accuracy: 0.4221 - val_loss: 2.0711 - val_accuracy: 0.4472
Epoch 4/10
2358/2358 [==============================] - 10s 4ms/step - loss: 2.1928 - accuracy: 0.4270 - val_loss: 2.0542 - val_accuracy: 0.4538
Epoch 5/10
2358/2358 [==============================] - 10s 4ms/step - loss: 2.1757 - accuracy: 0.4313 - val_loss: 2.0356 - val_accuracy: 0.4512
Epoch 6/10
2358/2358 [==============================] - 12s 5ms/step - loss: 2.1638 - accuracy: 0.4344 - val_loss: 2.0298 - val_accuracy: 0.4524
Epoch 7/10
2358/2358 [==============================] - 11s 5ms/step - loss: 2.1520 - accuracy: 0.4354 - val_loss: 2.0241 - val_ac

In [12]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

1310/1310 [==============================] - 3s 2ms/step - loss: 1.9993 - accuracy: 0.4648
Test Accuracy: 46.48%
